# Your First Model

Build a working ML model in 10 lines of code. No configuration, no boilerplate.

**What you'll learn:**
- Load and profile a dataset
- Split data properly (and why it matters)
- Train a model with one line
- Check if it's any good
- Understand what the model learned
- Finalize with the "final exam" (assess)
- Save and reload a model

```bash
pip install "mlw[xgboost]"
```

> Requires **Python 3.10+**. Check with `python3 --version`.

In [1]:
import ml
ml.__version__

'4.0.0a1'

## 1. Load Data

mlw ships 7 built-in datasets. We'll start with **Iris** — 150 flowers, 4 measurements each, belonging to 3 species. A classic first dataset because it's small enough to understand completely.

In [2]:
data = ml.dataset("iris")
data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


Each row is one flower. The 4 numeric columns are measurements (sepal = outer leaf, petal = inner leaf). The `species` column is what we want to predict — that's our **target**.

## 2. Profile

Before building anything, look at what you have. `profile()` tells you the shape, the target distribution, and any potential problems.

In [3]:
ml.profile(data, "species")

── Profile [classification] ────────────
Rows:     150
Columns:  5
Target:   species
Balance:  33% minority class

  ! Small dataset (150 rows). ~90 train, ~30 test after split.



Key things to notice:
- **classification** — the library auto-detected that `species` is a category (3 species), not a number. No configuration needed.
- **33% minority class** — the 3 species are evenly distributed (50 each). That's good — no class is underrepresented.
- **Small dataset warning** — 150 rows is tiny. The library warns you because small datasets mean less reliable results. Good to know upfront.

## 3. Split — Why Not Train on Everything?

Imagine studying for a test by memorizing the answer key. You'd ace *that* test, but fail any new questions. ML models can do the same thing — memorize the training data instead of learning the pattern.

The fix: hold out data the model never sees during training. Use it to check if the model actually learned.

`split()` divides your data into three parts:
- **train** (60%) — the model learns from this
- **valid** (20%) — your practice exam, check performance here, iterate freely
- **test** (20%) — the final exam, use once at the very end

In [4]:
s = ml.split(data, "species", seed=42)
print(s)

── Split [species · classification] ────
Balance: 33% minority class
Train:      90  (60%)
Valid:      30  (20%)
Dev:       120  (80%)
Test:       30  (20%)
Seed:   42



90 flowers for training, 30 for validation, 30 for the final test. The `Dev` row (120 = train + valid) is for later — once you've finished experimenting, you retrain on all non-test data before the final exam.

`seed=42` is a **random seed** — it controls the randomness of the split. Same seed = same split, every time. This makes your work reproducible: anyone with your code gets the same results.

## 4. Fit — Train the Model

One line. Tell it which data to learn from and which column to predict.

In [5]:
model = ml.fit(s.train, "species", seed=42)
print(model)

── Model [xgboost · classification] ────
Target:   species
Features: 4
Rows:     90  (0.01s)
Seed:     42
Hash:     2abb5aa1



That's it. The library:
- Auto-selected **XGBoost** (a strong default algorithm — falls back to Random Forest if xgboost isn't installed)
- Auto-detected **classification** (predicting categories, not numbers)
- Used all 4 measurement columns as **features**
- Trained on **90 rows** in 0.01 seconds

The `Hash` is a fingerprint — it uniquely identifies this exact model.

## 5. Evaluate — Is It Any Good?

`evaluate()` is the **practice exam**. It runs the model on the validation data (which the model has never seen) and reports how well it did. Call it as many times as you want — this is where you experiment.

In [6]:
metrics = ml.evaluate(model, s.valid)
metrics

── Metrics [classification] ────────────
accuracy            0.9000   precision_macro  0.8993
f1_weighted         0.8982   recall_weighted  0.9000
f1_macro            0.8982   recall_macro     0.9000
precision_weighted  0.8993   roc_auc_ovr      0.9700
(0.01s)

The most important metric here is **accuracy: 0.90** — the model correctly identified the species 90% of the time on flowers it had never seen.

The other metrics are different ways to measure "how good":
- **accuracy** — percentage of correct predictions
- **f1** — balances precision and recall (useful when classes are uneven)
- **precision** — when it says "setosa", how often is it right?
- **recall** — of all actual setosas, how many did it find?
- **roc_auc_ovr** — how well does it rank predictions by confidence? (0.97 is excellent). OVR = "one-vs-rest", a strategy for multiclass problems.

For a first model with zero tuning, 90% accuracy is solid.

## 6. Predict — What Does the Model Actually Say?

Let's see the actual predictions next to the real answers.

In [7]:
import pandas as pd

preds = ml.predict(model, s.valid)
comparison = pd.DataFrame({
    "predicted": preds.values,
    "actual": s.valid["species"].values,
})
comparison.head()

,predicted,actual
0,virginica,virginica
1,setosa,setosa
2,virginica,virginica
3,versicolor,versicolor
4,versicolor,versicolor


All correct on the first 5 rows. But we know 3 out of 30 are wrong. Let's find them.

In [ ]:
# Which flowers did it get wrong?
comparison[comparison.predicted != comparison.actual]

All 3 mistakes involve the versicolor/virginica boundary — the two species that look most alike. Setosa (with its distinctively small petals) is easiest to classify.

## 7. Explain — What Did the Model Learn?

The model got 90% accuracy, but *how* did it decide? `explain()` shows which features mattered most.

In [8]:
ml.explain(model)

── Explain [xgboost] ───────────────────
  petal length (cm)  ████████████████████   62.9%
  petal width (cm)   ██████████             32.0%
  sepal length (cm)  █                       3.8%
  sepal width (cm)                           1.3%

**Petal measurements dominate** (95% of importance). Sepal measurements barely matter.

This makes biological sense — the three iris species are most easily distinguished by their petal shape. The model independently discovered what botanists have known for decades. When `explain()` output matches domain knowledge, it's a good sign your model is learning real patterns, not noise.

## 8. Finalize — Retrain on More Data

You've been evaluating on `s.valid` — your practice exam. Now that you're happy with the model, retrain on `s.dev` (train + valid combined) to give it more data before the final test.

This is the `.dev` property in action — it exists specifically for this step.

In [ ]:
final = ml.fit(s.dev, "species", seed=42)
print(final)

## 9. Assess — The Final Exam

`assess()` is the **final exam**. Unlike `evaluate()` (which you can call freely), `assess()` is meant to be called **once**, on the test set that nobody — not you, not the model — has touched during development.

Why the distinction? If you keep tweaking your model based on test-set results, you're effectively training on the test set. `assess()` warns you if you call it more than once, keeping you honest.

In [ ]:
verdict = ml.assess(final, test=s.test)
verdict

This is the number that matters. The model's performance on completely unseen data — no peeking, no tweaking. Report this number in your paper, your presentation, or your Slack message to the team.

## 10. Save and Load — Keep Your Work

Save the model to disk. Load it later — in a different script, a different day, on a different machine.

In [ ]:
ml.save(final, "iris_model.ml")

# Later...
loaded = ml.load("iris_model.ml")
print(loaded)

In [ ]:
# Loaded model produces identical predictions
loaded_preds = ml.predict(loaded, s.test)
original_preds = ml.predict(final, s.test)
print(f"Predictions match: {(loaded_preds == original_preds).all()}")

Models are saved using [skops](https://skops.readthedocs.io/) — a secure format that doesn't execute arbitrary code on load (unlike pickle). The `.ml` extension is added automatically.

---

## Recap

The complete workflow, 10 functions:

```python
data    = ml.dataset("iris")                  # load
prof    = ml.profile(data, "species")          # inspect
s       = ml.split(data, "species", seed=42)   # split
model   = ml.fit(s.train, "species", seed=42)  # train
metrics = ml.evaluate(model, s.valid)           # practice exam (repeat freely)
imp     = ml.explain(model)                     # understand

final   = ml.fit(s.dev, "species", seed=42)    # retrain on more data
verdict = ml.assess(final, test=s.test)         # final exam (once)
ml.save(final, "model.ml")                      # save
loaded  = ml.load("model.ml")                   # load
```

No configuration files. No boilerplate. No classes to inherit from.

**What's next?** In [02_explore.ipynb](02_explore.ipynb), we'll use a bigger dataset and learn how to find the best algorithm — not just use the default.